# Notebook to fit y0

The idea behind this is similar to the idea in DCT and here https://doi.org/10.1107/S1600576724009634, where Friedel pairs are used to locate where diffraction spots come from in space. In those cases we use peaks that are 180 degrees apart. This notebook is looking for peaks that are separated by twotheta. These are the peaks we use in the friedel_rois macro at ID11 that aligns grains on the centre of rotation.

The pairs we use will have:
- eta -> -eta
- tth -> tth
- gve -> -gve

Jon Wright. March 2025.

In [ ]:
import ImageD11.friedel_pairs as fp


In [ ]:
# python environment stuff
IMAGED11_PATH = None  # means do not use git, otherwise "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = None  # None means guess, or you can specify a folder for the checkout

dset_path = 'si_cube_test/processed/Si_cube/Si_cube_S3DXRD_nt_moves_dty/Si_cube_S3DXRD_nt_moves_dty_dataset.h5'

par_file = None
# manual override of y0 (ignores any value saved in dataset)
y0_manual = None
# or y0_manual = 0.0 (typical NSCOPE) or 13.5 (typical TDXRD)
is_half_scan = False
# need to know for automatic y0 guessing from scan ranges.

gvtol = 0.002     # value is often OK

In [ ]:
if IMAGED11_PATH is not None:
    exec(open('/data/id11/nanoscope/install_ImageD11_from_git.py').read())
    PYTHONPATH=setup_ImageD11_from_git(CHECKOUT_PATH, IMAGED11_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.spatial
import ImageD11.sinograms.dataset
from ImageD11.sinograms.geometry import recon_bins
from tqdm.autonotebook import tqdm
from scipy.optimize import curve_fit

%matplotlib ipympl

In [ ]:
ds = ImageD11.sinograms.dataset.load(dset_path)
print(ds)

In [ ]:
if par_file is not None:
    # only change if ds has no parfile
    if not hasattr(ds, 'parfile') or ds.parfile is None:
        ds.parfile = par_file
        ds.save()

In [ ]:
cf_4d = ds.get_cf_4d()
ds.update_colfile_pars(cf_4d)
print(cf_4d.nrows/1e6, "million peaks read in")

The next cell is locating the Friedel pairs. It seems to need about 1 second per million peaks.

In [ ]:
ip, im = fp.find_pairs( cf_4d, gvtol=gvtol, doplot=True )
print('Got',len(ip),'pairs from',cf_4d.nrows,'peaks, fraction paired =',len(ip)*2/cf_4d.nrows )

In [ ]:
# Priority order (decreasing)
# manual override in first cell
# ds attribute

# fails if no y0 provided at all

if y0_manual is not None:
    print('Using manually supplied y0')
    y0_guess = y0_manual
else:
    if hasattr(ds, 'y0') and ds.y0 is not None:
        print('Using ds.y0')
        y0_guess = ds.y0
    else:
        # not using manual y0, and couldn't find one. guess it from ymax/ymin
        if not is_half_scan:
            y0_guess = (ds.ymax + ds.ymin)/2
        else:
            # half scan, not sure
            raise ValueError('You must manually supply a y0 value in the first cell to continue.')
print('Using y0:', y0_guess)

Now fit the positions with the y0 guess. Should be faster than finding the pairs.

In [ ]:
sx, sy = fp.locate_pairs( cf_4d, (ip,im), y0 = y0_guess )
bin_edges_guess, bin_centres_guess = recon_bins(ds.ybincens, ds.ymin, ds.ystep, y0_guess)
hist_guess = np.histogram2d(sx, sy, bins=bin_edges_guess)[0]

In [ ]:
fig, ax = plt.subplots(layout='constrained', figsize=(8,8))
ax.pcolormesh(bin_edges_guess, bin_edges_guess, hist_guess)
ax.set_aspect(1)
ax.set(title=f'y0 guess: {y0_guess}', xlabel='Sample Y axis -->', ylabel='Sample X axis -->')
plt.show()

In [ ]:
# now we try this with a range of y0 guesses, and look for peaks in the standard deviation

In [ ]:
# guess += 5 from y0_guess, you can change this as needed
y0_min = y0_guess - 5
y0_max = y0_guess + 5
n_y0 = 200
y0s = np.linspace(y0_min, y0_max, n_y0)

# choose how many peaks to use to fit - this is faster than using the full array
npks = 500_000 

In [ ]:
%%time

best_y0, y0s, stdevs = fp.fit_y0(cf_4d, (ip, im), y0s, npks=npks, doplot=True, fit_window=30)

In [ ]:
# take the results of the Laplace fit:
y0_final = best_y0
# or manually override from your interpretation of the plot:
# y0_final = 0
print(y0_final)

In [ ]:
sx, sy = fp.locate_pairs( cf_4d, (ip,im), y0 = y0_final )
bin_edges_final, bin_centres_final = recon_bins(ds.ybincens, ds.ymin, ds.ystep, y0_final)
hist_final = np.histogram2d(sx, sy, bins=bin_edges_final)[0]

In [ ]:
fig, ax = plt.subplots(layout='constrained', figsize=(8,8))
ax.pcolormesh(bin_edges_final, bin_edges_final, hist_final)
ax.set_aspect(1)
ax.set(title=f'y0 final: {y0_final}', xlabel='Sample Y axis -->', ylabel='Sample X axis -->')
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, layout='constrained', figsize=(12,6), sharex=True, sharey=True)
axs[0].pcolormesh(bin_edges_guess, bin_edges_guess, hist_guess)
axs[0].set_aspect(1)
axs[0].set(title=f'y0 guess: {y0_guess}')
axs[1].pcolormesh(bin_edges_final, bin_edges_final, hist_final)
axs[1].set_aspect(1)
axs[1].set(title=f'y0 final: {y0_final}')
fig.supxlabel('Sample Y axis -->')
fig.supylabel('Sample X axis -->')
plt.show()

In [ ]:
# save final result to disk
ds.y0 = y0_final
ds.save()